pytorchを使用して、重みWを特異値分解して、rankを指定して、重みWを再構成する。
またMSEをとって精度低下具合をみる。

In [1]:
import torch
import torch.nn as nn

# ============================================================
# 目的: SVD で重みを低ランク近似し、出力誤差を測る
# ============================================================
# 流れ:
#   1. 元の Linear 層で出力 y_original を計算
#   2. W を SVD 分解
#   3. 上位 r 個の特異値だけ使って W_r を再構成
#   4. W_r で出力 y_approx を計算し、差を MSE で評価

torch.manual_seed(42)  # 毎回同じ乱数 → 結果を再現できる

# MNIST 想定: 784次元 → 512次元の全結合層
layer = nn.Linear(784, 512)

# ダミー入力: 32枚 × 784次元（MNIST を flatten したイメージ）
x = torch.randn(32, 784)

# --- ステップ1: 元の出力 ---
y_original = layer(x)  # shape: (32, 512)

# --- ステップ2: 重み・バイアスを取り出して SVD ---
W = layer.weight.data  # (512, 784)
b = layer.bias.data    # (512,)
U, S, Vh = torch.linalg.svd(W, full_matrices=False)

print("W shape:", W.shape)
print("b shape:", b.shape)
print("y_original shape:", y_original.shape)

# --- ステップ3: rank r で打ち切って W_r を作る ---
r = 64  # 512 個の特異値のうち、上位 64 個だけ使う

U_r = U[:, :r]    # (512, 64)  左特異ベクトルの先頭 r 列
S_r = S[:r]       # (64,)      特異値の先頭 r 個
Vh_r = Vh[:r, :]  # (64, 784)  右特異ベクトルの先頭 r 行

# W_r ≈ U_r @ diag(S_r) @ Vh_r  → shape は元の W と同じ (512, 784)
W_r = U_r @ torch.diag(S_r) @ Vh_r
print("W_r shape:", W_r.shape)

# --- ステップ4: 近似重みで出力し、誤差を計算 ---
# PyTorch の Linear は内部で y = x @ W.T + b と計算される
y_approx = x @ W_r.T + b

diff = y_original - y_approx  # 各要素の誤差

# MSE: 誤差の二乗平均（小さいほど近似精度が高い）
mse = torch.mean(diff ** 2)
# 最大絶対誤差: 最もズレた出力要素の大きさ
max_abs_error = torch.max(torch.abs(diff))

print("MSE:", mse.item())
print("Max abs error:", max_abs_error.item())
# rank=64 なので MSE=0 にはならない（情報を捨てているため）

W shape: torch.Size([512, 784])
b shape: torch.Size([512])
y_original shape: torch.Size([32, 512])
W_r shape: torch.Size([512, 784])
MSE: 0.22959685325622559
Max abs error: 1.961763858795166
